In [4]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Loading real-world job posting from Monster.com dataset...")
try:
    jobs_df = pd.read_csv('resumes.csv')
except FileNotFoundError:
    jobs_df = pd.read_csv('monster_com-job_sample.csv')

jobs_df.columns = [c.lower().replace(' ', '_') for c in jobs_df.columns]
tech_jobs = jobs_df[jobs_df['job_description'].astype(str).str.contains('Python|Software|Developer|Data', case=False, na=False)]

raw_jd = tech_jobs.iloc[0]['job_description'] if len(tech_jobs) > 0 else jobs_df.iloc[0]['job_description']
target_jd = re.sub(r'<[^>]+>', ' ', str(raw_jd))

print(f"\n TARGET JOB ROLE SELECTED: Software / Data Engineering Role")
print(f"JD Preview: {target_jd[:250]}...\n")
print("="*80)

candidates = [
    {
        "name": "Candidate A (Senior Full-Stack Dev)",
        "resume": "Experienced Software Engineer with 6 years in Python, Java, SQL, AWS, cloud architecture, and machine learning. Proven track record in building scalable APIs and database optimization."
    },
    {
        "name": "Candidate B (Data & AI Specialist)",
        "resume": "Data Scientist and Python developer specializing in machine learning, SQL, data analysis, TensorFlow, and statistical modeling. Strong background in data visualization and reporting."
    },
    {
        "name": "Candidate C (Junior Web Developer)",
        "resume": "Junior web developer familiar with HTML, CSS, JavaScript, and basic Python. Eager to learn database management and backend systems."
    },
    {
        "name": "Candidate D (Project Manager / Non-Tech)",
        "resume": "Experienced Project Manager with expertise in Agile methodologies, Scrum, team leadership, budget forecasting, and Jira client communication."
    }
]

all_texts = [target_jd] + [c["resume"] for c in candidates]

vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(all_texts)

similarity_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()

core_skills = {"python", "sql", "java", "aws", "machine learning", "cloud", "javascript", "data analysis", "apis"}

results = []
for i, candidate in enumerate(candidates):
    resume_text_lower = candidate["resume"].lower()
    
    matched = [skill for skill in core_skills if skill in resume_text_lower]
    missing = [skill for skill in core_skills if skill not in resume_text_lower]
    
    results.append({
        "Candidate Name": candidate["name"],
        "Match Score (%)": round(similarity_scores[i] * 100, 2),
        "Matched Competencies": ", ".join(matched).title() if matched else "None",
        "Missing Core Skills (Skill Gap)": ", ".join(missing).title() if missing else "None"
    })

report_df = pd.DataFrame(results).sort_values(by="Match Score (%)", ascending=False).reset_index(drop=True)

print(" AUTOMATED CANDIDATE SCREENING & RANKING REPORT")
print("="*80)
display(report_df)

Loading real-world job posting from Monster.com dataset...

 TARGET JOB ROLE SELECTED: Software / Data Engineering Role
JD Preview: TeamSoft is seeing an IT Support Specialist to join our client in Madison, WI. The ideal candidate must have at least 6 years of experience in the field. They need to be familiar with a variety of the field's concepts, practices, and procedures as th...

 AUTOMATED CANDIDATE SCREENING & RANKING REPORT


,Candidate Name,Match Score (%),Matched Competencies,Missing Core Skills (Skill Gap)
0,Candidate D (Project Manager / Non-Tech),2.58,None,"Cloud, Machine Learning, Java, Python, Apis, J..."
1,Candidate C (Junior Web Developer),1.65,"Java, Python, Javascript","Cloud, Machine Learning, Apis, Sql, Data Analy..."
2,Candidate A (Senior Full-Stack Dev),1.44,"Cloud, Machine Learning, Java, Python, Apis, S...","Javascript, Data Analysis"
3,Candidate B (Data & AI Specialist),0.35,"Machine Learning, Python, Sql, Data Analysis","Cloud, Java, Apis, Javascript, Aws"


In [5]:
# Read the large dataset
df_big = pd.read_csv('resumes.csv')

# Keep only the first 500 rows to make the file size tiny for GitHub!
df_small = df_big.head(500)

# Save it back over the old file
df_small.to_csv('resumes.csv', index=False)
print("Done! Your CSV file is now tiny and ready for GitHub.")

Done! Your CSV file is now tiny and ready for GitHub.
